# ERS/ERD Add-on (After Preprocessing)
This section assumes you already executed **Load**, **Preprocess**, **Epoch**, **Autoreject**, and **ERP** cells,
and you have an `epochs` object with event IDs for left/right (e.g., `{'left':1, 'right':2}`).

The code below computes ERS/ERD time-courses with four methods (Hilbert, Morlet, Multitaper, STFT/Welch),
does baseline normalization (percent and dB), provides per-trial traces, detects onsets via multiple methods,
and plots time-courses and topomaps. The multitaper TFR plot is kept as requested.


In [ ]:

import numpy as np
import pandas as pd
import mne
from ersd_pipeline import run_ersd_bundle, plot_ersd_timeseries, plot_topomaps, plot_tfr_multitaper

# ---------- Parameters (edit as needed) ----------
BASELINE = (-2.0, -1.0)         # trial-wise baseline window
ANALYSIS = (-4.0, 6.0)          # analysis window (epochs should already cover this)
BANDS = {'mu': (8,12), 'beta': (13,30)}
CHANNELS_OF_INTEREST = ('C3','C4')

# TFR parameters (typical settings)
FREQS = np.arange(5.0, 40.1, 1.0)
MORLET_N_CYCLES = None          # None -> np.linspace(3,7,len(FREQS))
MT_N_CYCLES = 5.0               # 5 cycles in time window
MT_TIME_BW = 4.0                # ~7 tapers (2*NW-1)

# STFT/Welch sliding window
STFT_WIN_S = 0.5                # 500 ms
STFT_STEP_S = 0.05              # 50 ms

# Onset detection
ONSET_MIN_DUR = 0.1             # 100 ms sustain
ONSET_ALPHA = 0.05              # t-test FDR level
ONSET_K = 2.0                   # baseline SD multiplier

# Smoothing
SMOOTH_FWHM = 0.1               # seconds; set 0 to disable
UNITS = ('percent','db')        # stored outputs


In [ ]:

# ---------- Split epochs by condition ----------
event_ids = epochs.event_id
epochs_by_cond = {name: epochs[name] for name in event_ids.keys()}
print("Conditions:", list(epochs_by_cond.keys()))
print(epochs)


In [ ]:

# ---------- Run the ERS/ERD bundle ----------
out = run_ersd_bundle(
    epochs_by_cond,
    bands=BANDS,
    baseline=BASELINE,
    smooth_fwhm=SMOOTH_FWHM,
    unit_modes=UNITS,
    tfr_freqs=FREQS,
    morlet_n_cycles=MORLET_N_CYCLES,
    mt_n_cycles=MT_N_CYCLES,
    mt_time_bandwidth=MT_TIME_BW,
    stft_win_s=STFT_WIN_S,
    stft_step_s=STFT_STEP_S,
    channels_of_interest=CHANNELS_OF_INTEREST,
    onset_min_dur=ONSET_MIN_DUR,
    onset_alpha=ONSET_ALPHA,
    onset_k=ONSET_K,
    erd_win=(0.0, 2.0),
    ers_win=(2.0, 4.0),
    do_topomaps=True,
    return_all_matrices=False
)
ts = out['timeseries']
on = out['onsets']
print(ts.head())
print(on.head())

# Save CSVs
ts_path = "/mnt/data/ersd_timeseries.csv"
on_path = "/mnt/data/ersd_onsets.csv"
ts.to_csv(ts_path, index=False)
on.to_csv(on_path, index=False)
print("Saved:", ts_path, on_path)


In [ ]:

# ---------- Plots ----------
# A) Per-trial + mean time-courses (ERS/ERD % and dB) for each method
for cond in sorted(out['timeseries']['condition'].unique()):
    for band in BANDS.keys():
        for method in ('hilbert','morlet','multitaper','stft_welch'):
            for unit in UNITS:
                try:
                    plot_ersd_timeseries(out['timeseries'], cond, band, channels=CHANNELS_OF_INTEREST,
                                         method=method, unit=unit, shade_baseline=BASELINE)
                except Exception as e:
                    print("Plot skip:", cond, band, method, unit, e)

# B) Topomaps (ERD% and ERS%)
for cond in sorted(out['timeseries']['condition'].unique()):
    for band in BANDS.keys():
        try:
            plot_topomaps(out['topomaps'], cond, band)
        except Exception as e:
            print("Topomap skip:", cond, band, e)


In [ ]:

# ---------- Keep the multitaper TFR plot (average over trials) ----------
for cond, ep in epochs_by_cond.items():
    _ = plot_tfr_multitaper(ep, FREQS, n_cycles=MT_N_CYCLES, time_bandwidth=MT_TIME_BW, picks=CHANNELS_OF_INTEREST)
